# S2 · Fundamentos de PySpark

**Integrante** · Leonardo Jesus Huaman Arhuata

**Dimensión** · Volumen de aeronaves en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

Práctica sobre el histórico Bronze del 4 de enero de 2026 en UTC.

## 1. Lectura, transformaciones y plan

Abrimos un Parquet con esquema explícito, filtramos por segundos Unix y convertimos `time` a un timestamp UTC. La regla de elegibilidad es exploratoria.

In [2]:
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructType

bronze = Path("/opt/s01-arquitectura/data/2026")
bloque = bronze / "01" / "2026-01-01_2026-01-05.parquet"
dia_utc = date(2026, 1, 4)
inicio_utc = datetime.combine(dia_utc, time.min, tzinfo=timezone.utc)
fin_utc = inicio_utc + timedelta(days=1)

spark = (
    SparkSession.builder.master("local[2]")
    .appName("AeroStreamPeru-S02")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 15:14:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
esquema = (
    StructType()
    .add("time", LongType())
    .add("icao24", StringType())
    .add("lat", DoubleType())
    .add("lon", DoubleType())
    .add("onground", BooleanType())
    .add("velocity", DoubleType())
)
estados = spark.read.schema(esquema).parquet(str(bloque))
estados_del_dia = (
    estados.filter(
        (F.col("time") >= int(inicio_utc.timestamp()))
        & (F.col("time") < int(fin_utc.timestamp()))
    )
    .withColumn("captura_utc", F.timestamp_seconds("time"))
    .withColumn("icao24", F.lower(F.trim("icao24")))
)

identificador_valido = F.col("icao24").rlike(r"^[0-9a-f]{6}$")
posicion_en_zona = F.col("lat").between(-18.35, -0.04) & F.col("lon").between(-81.33, -68.65)
aeronave_en_vuelo = identificador_valido & posicion_en_zona & (~F.col("onground"))

estados_del_dia.printSchema()
estados_del_dia.explain(True)

root
 |-- time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- onground: boolean (nullable = true)
 |-- velocity: double (nullable = true)
 |-- captura_utc: timestamp (nullable = true)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(icao24, 'lower('trim('icao24)), None)]
+- Project [time#0L, icao24#1, lat#2, lon#3, onground#4, velocity#5, timestamp_seconds(time#0L) AS captura_utc#7]
   +- Filter ((time#0L >= cast(1767484800 as bigint)) AND (time#0L < cast(1767571200 as bigint)))
      +- Relation [time#0L,icao24#1,lat#2,lon#3,onground#4,velocity#5] parquet

== Analyzed Logical Plan ==
time: bigint, icao24: string, lat: double, lon: double, onground: boolean, velocity: double, captura_utc: timestamp
Project [time#0L, lower(trim(icao24#1, None)) AS icao24#8, lat#2, lon#3, onground#4, velocity#5, captura_utc#7]
+- Project [time#0L, icao24#1, lat#2, lon#3, onground#4, velocity#5, timesta

En `PushedFilters` aparecen los límites de `time`. Spark entrega el filtro temporal al lector Parquet. `filter` y `withColumn` construyen el plan, `explain` lo muestra y `show` ejecuta el trabajo para producir filas.

In [4]:
estados_del_dia.select("captura_utc", "icao24", "onground", "velocity").show(5)
print(f"Estados en el día de muestra: {estados_del_dia.count():,}")

+-------------------+------+--------+-----------------+
|        captura_utc|icao24|onground|         velocity|
+-------------------+------+--------+-----------------+
|2026-01-04 02:09:23|e80408|   false|             NULL|
|2026-01-04 02:09:24|e80408|   false|227.6657376748362|
|2026-01-04 02:09:25|e80408|   false|227.6657376748362|
|2026-01-04 02:09:26|e80408|   false|227.6657376748362|
|2026-01-04 02:09:27|e80408|   false|227.6657376748362|
+-------------------+------+--------+-----------------+
only showing top 5 rows
Estados en el día de muestra: 579,911


## 2. Mi indicador por ventana de diez minutos

El conteo usa `icao24` distinto por captura. Una captura observada que contiene estados, pero ninguna aeronave elegible, aporta cero. Un segundo sin registros queda fuera del promedio. El mínimo y máximo se calculan sobre los conteos por captura.

In [5]:
volumen_por_captura = estados_del_dia.groupBy("captura_utc").agg(
    F.countDistinct(F.when(aeronave_en_vuelo, F.col("icao24"))).alias("aeronaves_en_vuelo")
)
resumen_volumen = (
    volumen_por_captura
    .groupBy(F.window("captura_utc", "10 minutes").alias("ventana"))
    .agg(
        F.count("*").alias("capturas_observadas"),
        F.avg("aeronaves_en_vuelo").alias("volumen_promedio_t"),
        F.min("aeronaves_en_vuelo").alias("volumen_min_t"),
        F.max("aeronaves_en_vuelo").alias("volumen_max_t"),
    )
    .select(
        F.col("ventana.start").alias("inicio_utc"),
        "capturas_observadas",
        F.round("volumen_promedio_t", 2).alias("volumen_promedio_t"),
        "volumen_min_t",
        "volumen_max_t",
    )
    .orderBy("inicio_utc")
)
resumen_volumen.show(8, truncate=False)

[Stage 6:>                                                          (0 + 2) / 2]

+-------------------+-------------------+------------------+-------------+-------------+
|inicio_utc         |capturas_observadas|volumen_promedio_t|volumen_min_t|volumen_max_t|
+-------------------+-------------------+------------------+-------------+-------------+
|2026-01-04 00:00:00|598                |8.91              |7            |9            |
|2026-01-04 00:10:00|600                |8.49              |7            |10           |
|2026-01-04 00:20:00|600                |6.43              |5            |7            |
|2026-01-04 00:30:00|600                |6.71              |6            |7            |
|2026-01-04 00:40:00|600                |9.72              |7            |12           |
|2026-01-04 00:50:00|600                |12.54             |12           |13           |
|2026-01-04 01:00:00|599                |9.02              |2            |10           |
|2026-01-04 01:10:00|600                |5.97              |5            |7            |
+-------------------+

## 3. Ejercicio RDD

Agrupamos estados por aeronave durante la primera hora. `map` forma pares de identificador y conteo. `reduceByKey` suma los conteos. `takeOrdered` ejecuta la operación y devuelve cinco ejemplos.

In [6]:
primera_hora_en_vuelo = estados_del_dia.filter(
    aeronave_en_vuelo & (F.col("time") < int((inicio_utc + timedelta(hours=1)).timestamp()))
)
estados_por_aeronave = (
    primera_hora_en_vuelo.select("icao24").rdd
    .map(lambda fila: (fila.icao24, 1))
    .reduceByKey(lambda anterior, actual: anterior + actual)
)
for icao24, observaciones in estados_por_aeronave.takeOrdered(
    5, key=lambda par: (-par[1], par[0])
):
    print(f"{icao24}: {observaciones} estados")

[Stage 14:=============================>                            (1 + 1) / 2]

0c20a8: 3361 estados
aced2b: 2797 estados
0aca28: 2226 estados
0ac9dc: 1814 estados
0ac9c3: 1813 estados


## 4. Evidencia y defensa

Busca una aeronave que aparezca varias veces en la primera hora. Explica qué pasaría con el indicador si contaras estados en lugar de identificadores distintos por captura, y qué representa una ventana con menos de 600 capturas.

Para la evidencia individual, ejecuta tu notebook y registra tu salida de la celda de agregación y el `PushedFilters` del plan. La guía pide que la captura muestre reloj y usuario o perfil.

In [7]:
spark.stop()